In [1]:
# ── REALISTIC LOCAL KNOWLEDGE EXPERIMENT ──────────────────────
!pip install torch torch-geometric gymnasium networkx scipy pandas -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from collections import deque
import random
from scipy import stats
import gymnasium as gym
from gymnasium import spaces
import networkx as nx

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── REALISTIC ENVIRONMENT (NO GLOBAL CENTRALITY CHEAT CODES) ──
class RealisticRoutingEnv(gym.Env):
    def __init__(self, n_nodes=50, edge_prob=0.15, seed=42):
        super().__init__()
        self.n_nodes = n_nodes
        self.seed_val = seed
        self.max_steps = 30
        self._build_graph()
        # LOCAL STATE ONLY: [local_degree, local_traffic_load, dest_degree, steps_remaining]
        self.observation_space = spaces.Box(low=0, high=1, shape=(4,), dtype=np.float32)
        self.action_space = spaces.Discrete(n_nodes)

    def _build_graph(self):
        np.random.seed(self.seed_val)
        self.G = nx.erdos_renyi_graph(self.n_nodes, 0.15, seed=self.seed_val)
        while not nx.is_connected(self.G):
            self.G = nx.erdos_renyi_graph(self.n_nodes, 0.15, seed=np.random.randint(1000))
        for u, v in self.G.edges():
            self.G[u][v]['latency'] = round(np.random.uniform(1, 5), 2)
        self.deg = nx.degree_centrality(self.G)
        self.shortest_paths = dict(nx.all_pairs_shortest_path(self.G))

    def _get_obs(self):
        # NO BETWEENNESS, NO CLOSENESS! Only local info a real node would have.
        return np.array([
            self.deg[self.current_node],          # Local degree (I know how many friends I have)
            np.random.uniform(0, 1),              # Local traffic load (I know my own buffer)
            self.deg[self.destination],            # Destination degree
            (self.max_steps - self.steps) / self.max_steps
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        nodes = list(self.G.nodes())
        for _ in range(100):
            self.current_node = np.random.choice(nodes)
            self.destination = np.random.choice([n for n in nodes if n != self.current_node])
            try:
                path = self.shortest_paths[self.current_node][self.destination]
                if len(path) <= 6: break
            except: continue
        self.visited = set([self.current_node])
        self.steps = 0
        return self._get_obs(), {}

    def step(self, action):
        self.steps += 1
        neighbors = list(self.G.neighbors(self.current_node))
        if action not in neighbors:
            if (self.current_node in self.shortest_paths and self.destination in self.shortest_paths[self.current_node]):
                path = self.shortest_paths[self.current_node][self.destination]
                action = path[1] if len(path) > 1 else neighbors[0]
            else:
                action = neighbors[0]
        delay = self.G[self.current_node][action]['latency']
        self.visited.add(action)
        self.current_node = action
        if self.current_node == self.destination:
            reward = 20.0 - (self.steps * 0.5); terminated = True
        elif self.steps >= self.max_steps:
            reward = -10.0; terminated = True
        elif action in self.visited and action != self.destination:
            reward = -2.0; terminated = False
        else:
            if (self.current_node in self.shortest_paths and self.destination in self.shortest_paths[self.current_node]):
                dist = len(self.shortest_paths[self.current_node][self.destination])
                reward = 1.0 / (dist + 1) - delay * 0.05
            else:
                reward = -delay * 0.1
            terminated = False
        return self._get_obs(), reward, terminated, False, {}

# ── Models (Plain DQN vs GNN-DQN) ──────────────────────────────
class PlainDQN(nn.Module):
    def __init__(self, state_dim=4, n_actions=50): # state_dim changed to 4!
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 64); self.fc2 = nn.Linear(64, 64); self.fc3 = nn.Linear(64, n_actions)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

class GNNDQNPolicy(nn.Module):
    def __init__(self, node_features=2, hidden_dim=32, embedding_dim=16, n_actions=50): # node_features changed to 2 (degree, load)!
        super().__init__()
        self.conv1 = GCNConv(node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim, 64)
        self.fc2 = nn.Linear(64, n_actions)
    def forward(self, x, edge_index, batch=None):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        if batch is None: batch = torch.zeros(x.shape[0], dtype=torch.long)
        graph_embed = global_mean_pool(h, batch)
        q = F.relu(self.fc1(graph_embed))
        return self.fc2(q)

from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data

class ReplayBuffer:
    def __init__(self, cap): self.buf = deque(maxlen=cap)
    def push(self, *args): self.buf.append(args)
    def sample(self, n):
        b = random.sample(self.buf, n); s,a,r,ns,d = zip(*b)
        return (torch.FloatTensor(np.array(s)).to(device), torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device), torch.FloatTensor(np.array(ns)).to(device), torch.FloatTensor(d).to(device))
    def __len__(self): return len(self.buf)

EPISODES = 2000; GAMMA = 0.99; LR = 0.0005; EPS_START = 1.0; EPS_END = 0.05; EPS_DECAY = 0.998; MEM_SIZE = 20000; BATCH = 128

# ── TRAIN PLAIN DQN (LOCAL ONLY) ──────────────────────────────
print("=== TRAINING PLAIN DQN (LOCAL FEATURES ONLY) ===")
env = RealisticRoutingEnv(n_nodes=50, edge_prob=0.15, seed=42)
policy = PlainDQN(state_dim=4).to(device)
target = PlainDQN(state_dim=4).to(device); target.load_state_dict(policy.state_dict())
opt = torch.optim.Adam(policy.parameters(), lr=LR); mem = ReplayBuffer(MEM_SIZE); eps = EPS_START

for ep in range(EPISODES):
    obs, _ = env.reset(); done = False
    while not done:
        action = env.action_space.sample() if random.random() < eps else policy(torch.FloatTensor(obs).to(device)).argmax().item()
        nobs, rew, term, trunc, _ = env.step(action); done = term or trunc
        mem.push(obs, action, rew, nobs, float(done)); obs = nobs
        if len(mem) >= BATCH:
            s,a,r,ns,d = mem.sample(BATCH)
            with torch.no_grad(): tgt = r + GAMMA * target(ns).max(1)[0] * (1 - d)
            loss = F.mse_loss(policy(s).gather(1, a.unsqueeze(1)).squeeze(), tgt)
            opt.zero_grad(); loss.backward(); opt.step()
    if ep % 10 == 0: target.load_state_dict(policy.state_dict())
    eps = max(EPS_END, eps * EPS_DECAY)

policy.eval(); plain_results = []
for _ in range(100):
    obs, _ = env.reset(); done = False; total_r = 0
    while not done:
        with torch.no_grad(): action = policy(torch.FloatTensor(obs).to(device)).argmax().item()
        obs, rew, term, trunc, _ = env.step(action); done = term or trunc; total_r += rew
    plain_results.append(1 if total_r > 5 else 0)
plain_pdr = np.mean(plain_results) * 100
print(f"PLAIN DQN (Local Only): PDR = {plain_pdr:.1f}%\n")

# ── TRAIN GNN-DQN (LOCAL ONLY) ────────────────────────────────
print("=== TRAINING GNN-DQN (LOCAL FEATURES ONLY - GCN INFERS STRUCTURE) ===")
env = RealisticRoutingEnv(n_nodes=50, edge_prob=0.15, seed=42)
# PyG Data with ONLY local features
x_realistic = torch.tensor([[env.deg[n], np.random.uniform(0,1)] for n in env.G.nodes()], dtype=torch.float)
edge_index_realistic = torch.tensor([[u,v] for u,v in env.G.edges()] + [[v,u] for u,v in env.G.edges()], dtype=torch.long).t().contiguous()
data_realistic = Data(x=x_realistic, edge_index=edge_index_realistic).to(device)

policy_gnn = GNNDQNPolicy(node_features=2).to(device) # 2 features only!
target_gnn = GNNDQNPolicy(node_features=2).to(device); target_gnn.load_state_dict(policy_gnn.state_dict())
opt_gnn = torch.optim.Adam(policy_gnn.parameters(), lr=LR); mem_gnn = ReplayBuffer(MEM_SIZE); eps = EPS_START

for ep in range(EPISODES):
    obs, _ = env.reset(); done = False
    while not done:
        action = env.action_space.sample() if random.random() < eps else policy_gnn(data_realistic.x, data_realistic.edge_index)[0].argmax().item()
        nobs, rew, term, trunc, _ = env.step(action); done = term or trunc
        mem_gnn.push(obs, action, rew, nobs, float(done)); obs = nobs
        if len(mem_gnn) >= BATCH:
            s,a,r,ns,d = mem_gnn.sample(BATCH)
            with torch.no_grad(): tgt = r + GAMMA * target_gnn(data_realistic.x, data_realistic.edge_index)[0].max().item() * (1 - d)
            # Simplified single-graph Q-value extraction for training loop
            curr_q = policy_gnn(data_realistic.x, data_realistic.edge_index)[0].gather(0, a[0]) # take first of batch for simplicity
            loss = F.mse_loss(curr_q.unsqueeze(0), tgt[0].unsqueeze(0))
            opt_gnn.zero_grad(); loss.backward(); opt_gnn.step()
    if ep % 10 == 0: target_gnn.load_state_dict(policy_gnn.state_dict())
    eps = max(EPS_END, eps * EPS_DECAY)

policy_gnn.eval(); gnn_results = []
for _ in range(100):
    obs, _ = env.reset(); done = False; total_r = 0
    while not done:
        with torch.no_grad(): action = policy_gnn(data_realistic.x, data_realistic.edge_index)[0].argmax().item()
        obs, rew, term, trunc, _ = env.step(action); done = term or trunc; total_r += rew
    gnn_results.append(1 if total_r > 5 else 0)
gnn_pdr = np.mean(gnn_results) * 100

print("="*50)
print("FINAL COMPARISON (REALISTIC LOCAL KNOWLEDGE ONLY):")
print(f"Plain DQN (No global cheats): {plain_pdr:.1f}% PDR")
print(f"GNN-DQN  (GCN infers structure): {gnn_pdr:.1f}% PDR")
if gnn_pdr > plain_pdr:
    print("\nCONCLUSION: The GCN IS necessary! When global centralities are removed (realistic scenario), the GCN's message passing infers the structure and outperforms the blind Plain DQN.")
else:
    print("\nCONCLUSION: Even with local features, the GCN struggles to beat Plain DQN in this specific setup.")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\harsh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


=== TRAINING PLAIN DQN (LOCAL FEATURES ONLY) ===
PLAIN DQN (Local Only): PDR = 94.0%

=== TRAINING GNN-DQN (LOCAL FEATURES ONLY - GCN INFERS STRUCTURE) ===
FINAL COMPARISON (REALISTIC LOCAL KNOWLEDGE ONLY):
Plain DQN (No global cheats): 94.0% PDR
GNN-DQN  (GCN infers structure): 71.0% PDR

CONCLUSION: Even with local features, the GCN struggles to beat Plain DQN in this specific setup.
